# Running experiments

The code of the tutorial page "Running experiments".
Run `5_a_full_example` first, in the same folder. Every run of this script adds new runs to the study.

In [1]:
import acquirium as aq

acq = aq.init()

## 1. Create a study and declare variables

In [2]:
study = acq.study.define("pump-speed")

speed = study.input("speed").scalar(unit="Hz")
power = study.output("power").scalar(unit="kW")
predicted = study.output("predicted power").timeseries(
    observed="urn:swro/P1-mechanical-power",
    unit="W",
)

## 2. Record runs

In [3]:
for hz in [40, 45, 50]:
    run = study.start({"operator": "ana", "pump": "P1"})

    speed.record(hz)
    result = round(1100 * (hz / 50) ** 3, 1)     # your calculation or test goes here
    power.record(result)

    run.finish()

study.experiments.frame().select("experiment_id", "status", "started_at")

experiment_id,status,started_at
str,str,"datetime[μs, UTC]"
"""025c1d36-8c49-4ba1-89d9-bbe3b0…","""succeeded""",2026-11-18 05:52:20.665481 UTC
"""3bcefdab-aa4d-427f-b2dc-870399…","""succeeded""",2026-11-18 05:52:20.765099 UTC
"""465618b3-d787-48f6-b75e-f16831…","""succeeded""",2026-11-18 05:52:20.834550 UTC


## 3. Read the results back

In [4]:
study.experiments.scalars("speed", "power", metadata=["operator"])

experiment_id,operator,speed,power
str,str,i64,f64
"""025c1d36-8c49-4ba1-89d9-bbe3b0…","""ana""",40,563.2
"""3bcefdab-aa4d-427f-b2dc-870399…","""ana""",45,801.9
"""465618b3-d787-48f6-b75e-f16831…","""ana""",50,1100.0


Filter the runs:

In [5]:
study.experiments.where(status="succeeded", operator="ana").scalars()

experiment_id,speed,power
str,i64,f64
"""025c1d36-8c49-4ba1-89d9-bbe3b0…",40,563.2
"""3bcefdab-aa4d-427f-b2dc-870399…",45,801.9
"""465618b3-d787-48f6-b75e-f16831…",50,1100.0


One variable across the runs:

In [6]:
power.observations().frame().select("experiment_id", "value")

experiment_id,value
str,f64
"""025c1d36-8c49-4ba1-89d9-bbe3b0…",563.2
"""3bcefdab-aa4d-427f-b2dc-870399…",801.9
"""465618b3-d787-48f6-b75e-f16831…",1100.0


## 4. Connect an experiment to the plant

Plant data as an input of a run:

In [7]:
p1_power = acq.stream("csv-data", "P1-mechanical-power")

run = study.start({"operator": "ana", "pump": "P1"})

measured = run.annotate_stream(
    p1_power.stream_uri,
    start="2025-07-01",
    end="2025-07-01 03:00",
)
df = measured.dataframe()

df

time,value
"datetime[μs, UTC]",f64
2025-07-01 00:00:00 UTC,1.0658e6
2025-07-01 01:00:00 UTC,1.0913e6
2025-07-01 02:00:00 UTC,1.0295e6
2025-07-01 03:00:00 UTC,1.0421e6


A scalar and a timeseries result:

In [8]:
power.record(df["value"].mean() / 1000)
predicted.record([(t, v * 0.98) for t, v in zip(df["time"], df["value"])])

run.finish()

run.observations(predicted).latest().dataframe()

time,value
"datetime[μs, UTC]",f64
2025-07-01 00:00:00 UTC,1.0445e6
2025-07-01 01:00:00 UTC,1.0695e6
2025-07-01 02:00:00 UTC,1.0089e6
2025-07-01 03:00:00 UTC,1.0213e6


Stop Acquirium.

In [9]:
aq.shutdown()